# Multigrid Track — Phase 0 Validation Notebook

Reproduces every test of `multigrid_selfgravity_validation.pdf`: the shearing-box +
mesh-refinement infrastructure (orbital-advection toggle, interior-$x_1$ and annular
refinement policies, FARGO on refined meshes).

**Prerequisites**: the code built (`cmake .. && make -j8`; no FFT flag needed for this
notebook) and Julia packages `CairoMakie`, `IJulia`.

**Structure**
1. Helpers: running AthenaK, reading `.hst` and per-block `.bin`
2. Frame consistency: FARGO vs full-velocity (uniform grid)
3. Guard checks: the two policies must refuse illegal setups
4. FARGO on an annular refined ring vs its non-FARGO twin
5. The 20-orbit epicycle endurance test (the decisive one)
6. Mesh-layout visualization from `.bin` outputs
7. Playground

In [ ]:
const REPO   = expanduser("~/Library/CloudStorage/Dropbox/Research/code/athenak-multigrid")
const ATHENA = joinpath(REPO, "build", "src", "athena")
const INPUTS = joinpath(REPO, "inputs")
const RUN    = joinpath(REPO, "validation", "run")          # short shwave/ring tests
const RUNEPI = joinpath(REPO, "validation", "run_epicycle") # 20-orbit epicycle runs
mkpath(RUN); mkpath(RUNEPI)
isfile(ATHENA) || @warn "athena executable not found -- build first"
using CairoMakie, Printf
CairoMakie.activate!(type="png")
set_theme!(Theme(fontsize=13, Axis=(xgridcolor=(:gray, 0.25), ygridcolor=(:gray, 0.25))))
const C1, C2, C3 = "#2a78d6", "#eb6834", "#1baf7a";


In [ ]:
"Run athena on `input` (path relative to inputs/) in `dir` with overrides; return stdout."
function run_athena(input; overrides=String[], dir=RUN)
    cmd = Cmd(String[ATHENA, "-i", joinpath(INPUTS, input), overrides...])
    read(pipeline(Cmd(cmd; dir=dir); stderr=devnull), String)
end

"Run athena expecting a FATAL abort; return the fatal message lines."
function run_expect_fatal(input; overrides=String[], dir=RUN)
    out = read(pipeline(ignorestatus(Cmd(Cmd(String[ATHENA, "-i",
              joinpath(INPUTS, input), overrides...]); dir=dir)); stderr=devnull), String)
    lines = split(out, '\n')
    i = findfirst(l -> occursin("FATAL", l), lines)
    i === nothing ? error("expected a FATAL error but the run proceeded!") :
                    join(lines[i:min(i+3, length(lines))], '\n')
end

"Read an AthenaK .hst file into a matrix (rows = outputs)."
function read_hst(f)
    rows = Float64[]; ncol = 0
    for ln in eachline(f)
        startswith(strip(ln), "#") && continue
        v = parse.(Float64, split(ln)); ncol = length(v); append!(rows, v)
    end
    permutedims(reshape(rows, ncol, :))
end

"Read an AthenaK .bin file into per-MeshBlock records (multilevel-safe)."
function read_bin_blocks(filename)
    open(filename, "r") do io
        startswith(readline(io), "Athena binary output") || error("not an AthenaK bin")
        npre = parse(Int, split(readline(io), "=")[end])
        ph = Dict(String(strip(k)) => String(strip(v)) for (k, v) in
                  (split(readline(io), "=") for _ in 1:npre-1))
        locsize = parse(Int, ph["size of location"])
        varsize = parse(Int, ph["size of variable"])
        nvars = parse(Int, split(readline(io), "=")[end])
        vars = String.(split(readline(io))[2:end])
        hsize = parse(Int, split(readline(io), "=")[end])
        header = String(read(io, hsize))
        m = match(r"nghost\s*=\s*(\d+)", header)
        ng = parse(Int, m.captures[1])
        locT = locsize == 8 ? Float64 : Float32
        varT = varsize == 8 ? Float64 : Float32
        blocks = NamedTuple[]
        while !eof(io)
            idx = Int.(reinterpret(Int32, read(io, 24))) .- ng
            n1 = idx[2]-idx[1]+1; n2 = idx[4]-idx[3]+1; n3 = idx[6]-idx[5]+1
            logical = Int.(reinterpret(Int32, read(io, 16)))       # lx1,lx2,lx3,level
            geom = Float64.(reinterpret(locT, read(io, 6*locsize)))  # x1min..x3max
            raw = reinterpret(varT, read(io, n1*n2*n3*nvars*varsize))
            data = reshape(Float64.(raw), (n1, n2, n3, nvars))
            push!(blocks, (logical=logical, geom=geom, data=data))
        end
        (blocks=blocks, vars=vars)
    end
end;


## 1. Frame consistency — FARGO vs full-velocity, uniform grid

`<shearing_box> orbital_advection = false` switches to the full-velocity frame: the
solver integrates the shear directly, with full Coriolis + tidal source terms and
momentum/energy offsets in the shear-periodic wrap. Same physics, different variables —
frame-independent diagnostics must agree to truncation order.

In [ ]:
for (bn, ov) in (("shw_fargo", String[]),
                 ("shw_nofargo", ["shearing_box/orbital_advection=false"]))
    isfile(joinpath(RUN, bn * ".hydro.hst")) ||
        run_athena("shearing_box/hydro_incompress_shwave.athinput";
                   overrides=vcat("job/basename=$bn", ov))
end
a = read_hst(joinpath(RUN, "shw_fargo.hydro.hst"))
b = read_hst(joinpath(RUN, "shw_nofargo.hydro.hst"))
n = min(size(a, 1), size(b, 1))
@printf("mass : max rel diff = %.2e   (frame-independent, expect ~0)\n",
        maximum(abs.(a[1:n,3] .- b[1:n,3]))/maximum(abs.(a[1:n,3])))
@printf("x-KE : max rel diff = %.4f   (truncation gap between the two schemes)\n",
        maximum(abs.(a[1:n,7] .- b[1:n,7]))/maximum(abs.(a[1:n,7])))


## 2. Guard checks — both policies must refuse illegal setups

The policies are enforced with fatal errors, not conventions. Two runs that must abort:
a refined region whose blocks touch the shear-periodic $x_1$ boundary, and a
partial-$x_2$ ring with FARGO on.

In [ ]:
println("--- interior-x1 policy:")
println(run_expect_fatal("shearing_box/hydro_incompress_shwave_smr.athinput";
        overrides=["refined_region1/x1min=-0.24", "refined_region1/x1max=-0.13"]))
println("\n--- annular policy (partial ring + FARGO):")
println(run_expect_fatal("shearing_box/hydro_incompress_shwave_smr.athinput";
        overrides=["shearing_box/orbital_advection=true"]))


## 3. FARGO on an annular refined ring

The `_ring` input refines a full-$x_2$ level-1 annulus (interior in $x_1$). Every
$x_2$-face neighbor is then same-level, so the per-block shift kernel and the existing
communication run unchanged, level by level. Checks: exact mass conservation, and
agreement with the non-FARGO twin at the same truncation level as on uniform grids.

In [ ]:
for (bn, ov) in (("ring_fargo", String[]),
                 ("ring_nofargo", ["shearing_box/orbital_advection=false"]))
    isfile(joinpath(RUN, bn * ".hydro.hst")) ||
        run_athena("shearing_box/hydro_incompress_shwave_smr_ring.athinput";
                   overrides=vcat("job/basename=$bn", ov))
end
rf = read_hst(joinpath(RUN, "ring_fargo.hydro.hst"))
rn = read_hst(joinpath(RUN, "ring_nofargo.hydro.hst"))
n = min(size(rf, 1), size(rn, 1))
@printf("ring+FARGO mass drift        = %.2e   (expect exactly 0)\n",
        maximum(abs.(rf[:,3] .- rf[1,3])))
@printf("ring FARGO vs non-FARGO x-KE = %.4f   (uniform-grid gap was 3.3%%)\n",
        maximum(abs.(rf[1:n,7] .- rn[1:n,7]))/maximum(abs.(rf[1:n,7])))


## 4. The endurance test — epicycles for 20 orbits

`ipert=1` initializes a spatially **uniform** epicycle $v_x(0)=A$, an exact nonlinear
solution: $v_x = A\cos\kappa t$, $v_y' = -(A\kappa/2\Omega)\sin\kappa t$, with
$\kappa=\Omega$ at $q=3/2$. A uniform field must pass through the $y$-shift *exactly*,
so any indexing/communication defect at ring or level boundaries appears as $O(1)$
corruption — while phase/amplitude errors of the time integrator stay tiny and smooth.
The **amplitude invariant** $A(t)=\sqrt{v_x^2+(2\Omega/\kappa)^2v_y'^2}$ separates
numerical damping (drift in $A$) from phase error.

Note on units: `amp = 0.1` is a velocity in code units (`shwave.cpp` never multiplies
by $c_s$); with the ideal EOS at $p_0=\rho_0=1$, $c_s=\sqrt{5/3}$, so this epicycle is
$0.077\,c_s$ — the plot is normalized to `amp`, oscillating between $\pm1$.

In [ ]:
# ~1 min (uniform) + ~3 min (ring) if the runs don't exist yet
isfile(joinpath(RUNEPI, "epi_unif.hydro.hst")) ||
    run_athena("shearing_box/epicycle.athinput"; dir=RUNEPI)
isfile(joinpath(RUNEPI, "epi_ring.hydro.hst")) ||
    run_athena("shearing_box/epicycle_smr_ring.athinput"; dir=RUNEPI)

amp, Ω, q = 0.1, 1.0, 1.5
κ = sqrt(2*(2-q))*Ω
Torb = 2π/Ω
u = read_hst(joinpath(RUNEPI, "epi_unif.hydro.hst"))
r = read_hst(joinpath(RUNEPI, "epi_ring.hydro.hst"))
vx_u = u[:,4]./u[:,3]; vy_u = u[:,5]./u[:,3]; t_u = u[:,1]
vx_r = r[:,4]./r[:,3]; vy_r = r[:,5]./r[:,3]; t_r = r[:,1]
envel(vx, vy) = sqrt.(vx.^2 .+ (2Ω/κ)^2 .* vy.^2)
A_u, A_r = envel(vx_u, vy_u), envel(vx_r, vy_r)
ana(t) = amp .* cos.(κ .* t)
for (tag, t, v, A) in (("uniform   ", t_u, vx_u, A_u), ("ring+FARGO", t_r, vx_r, A_r))
    @printf("%s  vx∈[%+.5f, %+.5f]  amplitude drift %+.3f%%  max|vx-analytic|/amp = %.4f\n",
            tag, minimum(v), maximum(v), 100*(A[end]-A[1])/amp,
            maximum(abs.(v .- ana(t)))/amp)
end


In [ ]:
fig = Figure(size=(880, 620))
ax1 = Axis(fig[1,1]; xlabel="t / T_orb", ylabel="vₓ / amp",
           title="Epicyclic radial velocity over 20 orbits")
hlines!(ax1, [-1, 1]; color=(:gray, 0.55), linestyle=:dash)
lines!(ax1, t_u ./ Torb, vx_u ./ amp; color=C1, linewidth=1.2, label="uniform grid")
lines!(ax1, t_r ./ Torb, vx_r ./ amp; color=C3, linewidth=1.2, linestyle=:dash,
       label="annular ring + FARGO")
axislegend(ax1; position=:rt, framevisible=false, orientation=:horizontal)
ylims!(ax1, -1.35, 1.35)

sel_u = t_u ./ Torb .>= 18; sel_r = t_r ./ Torb .>= 18
tf = range(18Torb, t_u[end]; length=600)
ax2 = Axis(fig[2,1]; xlabel="t / T_orb", ylabel="vₓ / amp", height=150,
           title="last two orbits vs analytic amp·cos(κt)")
lines!(ax2, tf ./ Torb, ana(tf) ./ amp; color=C2, linewidth=2.5, label="analytic")
scatter!(ax2, t_u[sel_u] ./ Torb, vx_u[sel_u] ./ amp; color=C1, markersize=6,
         label="uniform")
scatter!(ax2, t_r[sel_r] ./ Torb, vx_r[sel_r] ./ amp; color=C3, marker=:rect,
         markersize=5, label="ring")
axislegend(ax2; position=:rt, framevisible=false, orientation=:horizontal)

ax3 = Axis(fig[3,1]; xlabel="t / T_orb", ylabel="A(t) / amp", height=150,
           title="amplitude invariant — flat means no numerical damping")
hlines!(ax3, [1.0]; color=(:gray, 0.55), linestyle=:dash)
lines!(ax3, t_u ./ Torb, A_u ./ amp; color=C1, linewidth=1.5, label="uniform")
lines!(ax3, t_r ./ Torb, A_r ./ amp; color=C3, linewidth=1.5, linestyle=:dash,
       label="ring + FARGO")
axislegend(ax3; position=:rb, framevisible=false, orientation=:horizontal)
fig


## 5. Mesh-layout visualization

Drawn directly from the per-block structure of the `.bin` outputs — the same reader you
will use for any refined-mesh analysis. Left: the interior patch (legal only without
FARGO). Right: the annular ring (FARGO-legal). Orange lines mark the shear-periodic
$x_1$ boundaries that refinement must avoid.

In [ ]:
function draw_blocks!(ax, fd; title="")
    lev_col = Dict(0 => :white, 1 => "#dbe9fb", 2 => "#b9d5f6")
    z0 = minimum(b.geom[5] for b in fd.blocks)
    for b in fd.blocks
        b.geom[5] ≈ z0 || continue          # one z-layer
        g = b.geom; lev = b.logical[4]
        poly!(ax, Rect2(g[1], g[3], g[2]-g[1], g[4]-g[3]);
              color=get(lev_col, lev, "#8fbdf0"), strokecolor=:black, strokewidth=0.8)
    end
    ax.title = title
end

f_patch = first(sort(filter(f -> occursin("shwave2_smr.hydro_w", f),
                            readdir(joinpath(RUN, "bin"); join=true))))
f_ring = first(sort(filter(f -> occursin("ring_fargo.hydro_w", f),
                           readdir(joinpath(RUN, "bin"); join=true))))
fig = Figure(size=(860, 430))
for (i, (f, ttl)) in enumerate(((f_patch, "patch (FARGO off only)"),
                                (f_ring, "annular ring (FARGO OK)")))
    ax = Axis(fig[1, i]; xlabel="x", ylabel=(i == 1 ? "y" : ""), aspect=DataAspect())
    draw_blocks!(ax, read_bin_blocks(f); title=ttl)
    vlines!(ax, [-0.25, 0.25]; color=C2, linewidth=3)
end
fig


## Playground

| override | effect |
|---|---|
| `shearing_box/orbital_advection=false` | full-velocity (non-FARGO) mode |
| `problem/ipert=1|2|3` | epicycle / vortical shwave / compressive shwave |
| `refined_region1/x1min=…` etc. | move/resize the refined region (policies enforced) |
| `problem/amp=…` | perturbation amplitude (code units, not $c_s$) |
| `time/tlim=…` | run length |

**Next (Phase 1):** `shear_periodic` boundary conditions in the multigrid driver,
validated against the FFT oracle — see `multigrid_selfgravity_validation.pdf` Sec. 5.